## Load Train/Val/Test Splits

In [1]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

## Filter: Keep Only Delivered Orders

In [2]:
train_df = train_df[train_df["order_status"] == "delivered"].reset_index(drop=True)
val_df = val_df[val_df["order_status"] == "delivered"].reset_index(drop=True)
test_df = test_df[test_df["order_status"] == "delivered"].reset_index(drop=True)

print(train_df.shape, val_df.shape, test_df.shape)

(67224, 18) (14676, 18) (14578, 18)


## Extract Safe Time-Based Features

In [3]:
for df in [train_df, val_df, test_df]:
    df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_weekday"] = df["order_purchase_timestamp"].dt.weekday
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

## Handle Missing Values (Join-Related)

In [4]:
for df in [train_df, val_df, test_df]:
    df["num_items"] = df["num_items"].fillna(0)
    df["total_price"] = df["total_price"].fillna(0)
    df["total_freight"] = df["total_freight"].fillna(0)
    df["total_payment_value"] = df["total_payment_value"].fillna(0)
    df["num_payment_methods"] = df["num_payment_methods"].fillna(0)

## One-Hot Encode Customer State

In [5]:
train_encoded = pd.get_dummies(train_df["customer_state"], prefix="state")
val_encoded = pd.get_dummies(val_df["customer_state"], prefix="state")
test_encoded = pd.get_dummies(test_df["customer_state"], prefix="state")

# محاذاة الأعمدة بين الثلاثة (val/test ممكن ينقصهم فئات موجودة بس بـ train)
val_encoded = val_encoded.reindex(columns=train_encoded.columns, fill_value=0)
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

train_df = pd.concat([train_df, train_encoded], axis=1)
val_df = pd.concat([val_df, val_encoded], axis=1)
test_df = pd.concat([test_df, test_encoded], axis=1)

## Build Final Feature Table (X, y)

In [6]:
feature_cols = ["num_items", "total_price", "total_freight", "total_payment_value", 
                "num_payment_methods", "purchase_month", "purchase_weekday", "purchase_hour"] + list(train_encoded.columns)

X_train = train_df[feature_cols]
y_train = train_df["is_late"]

X_val = val_df[feature_cols]
y_val = val_df["is_late"]

X_test = test_df[feature_cols]
y_test = test_df["is_late"]

print(X_train.shape, X_val.shape, X_test.shape)

(67224, 35) (14676, 35) (14578, 35)


## Save Feature Artifacts

In [7]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
X_val.to_csv("../data/processed/X_val.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved successfully!")

Saved successfully!


In [8]:
import json

with open("../models/feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

print("Saved feature columns:", len(feature_cols), "columns")

Saved feature columns: 35 columns
